In [0]:
%run /KKLLK/KKLLK/includes/KKLLKKKLLKCore 

In [0]:
from KKLLK.KKLLK.core import classBatch, classDataset, classLog, classDelta, classProcess
from datetime import datetime
from dateutil.relativedelta import relativedelta
env = classProcess().env

In [0]:
from KKLLK.KKLLK.core import classBatch, classDataset, classLog, classDelta

In [0]:
dbutils.widgets.text("batchId", "")
dbutils.widgets.text("entityId", "547868")

In [0]:
## Get widget parameters
try:  
  dbutils.widgets.get("batchId")
  batchId = getArgument("batchId")
except:
  pass  
dbutils.widgets.get("entityId")
entityId = getArgument("entityId")
if not entityId:
  dbutils.notebook.exit("Cannot continue processing without an entityId")

In [0]:
## Register the start of the processing
nobatch = classBatch(batchId, entityId)
nobatch.start(entityId)
## Get the batch Id
batchId = nobatch.getId()
## Get an instance of classFact and start the preProcess
dataset = classDataset(entityId)
dataset.preProcess()

In [0]:
## Set delta date variable to ensure that it is not instantiated 
deltaDate = dataset.deltaDate
print(deltaDate)

##### Tip: ...

In [0]:
baseJoinedWithDups = sql(f"""select  
KKLLK_{env}.function.hashValue(concat(CAST(api.dumbmeterNonHomenonaccounthome as STRING),api.dumbmeterNonHomeMeterSerialNumber,CAST(api.dumbmeterNonHomeMeterReadDate as STRING))) as id
,api.dumbmeterNonHomenonaccounthome as nonaccounthome
,api.dumbmeterNonHomeMeterSerialNumber as meterSerialNumber
,Cast(api.dumbmeterNonHomeMeterReadDate as STring) as NonHomeDate
,date_format(api.dumbmeterNonHomeMeterReadTimestamp,'yyyy-MM-dd HH:mm:SS')   as meterReadTimestamp
,api.dumbmeterNonHomeMeterReadType as meterReadType
,CAST(api.dumbmeterNonHomeMeterReadValue AS INTEGER) as meterReadValue
,api.dumbmeterNonHomeErrorCode as errorCode
,ROW_NUMBER() OVER (PARTITION BY api.dumbmeterNonHomenonaccounthome,api.dumbmeterNonHomeMeterSerialNumber, CAST(api.dumbmeterNonHomeMeterReadDate as date) ORDER BY api.dumbmeterNonHomeMeterReadDate desc, api.dumbmeterNonHomeValidInd desc) as RN  
,case 
                          WHEN api.dumbmeterNonHomeValidInd = 0 THEN 1 
                          WHEN unix_timestamp(api.dumbmeterNonHomeMeterReadDate)-unix_timestamp(now() - INTERVAL 24 months) < 1 THEN 1 
                          ELSE unix_timestamp(api.dumbmeterNonHomeMeterReadDate)-unix_timestamp(now() - INTERVAL 24 months) END  as ttl    
from datasetdumbmeterNonHomeFull api 
where KKLLKMetadataLoadDateTime >= CAST('{deltaDate}' as TIMESTAMP)
""")
baseJoinedWithDups.createOrReplaceTempView("baseJoinedSortDeltasInTheSameBatch")

baseJoined = spark.sql("""
                       SELECT  distinct id
                               ,nonaccounthome
                               ,meterSerialNumber
                               ,NonHomeDate
                               ,meterReadTimestamp
                               ,meterReadType
                               ,meterReadValue
                               ,errorCode
                               ,ttl  
                       FROM baseJoinedSortDeltasInTheSameBatch
                       WHERE RN = 1
                       """)  # IMPORTANT TO GET RID OF MULTIPLE MATCHES
#baseJoined.createOrReplaceTempView("test")

In [0]:

try:
  processedRows = dataset.postProcess(baseJoined)
  nobatch.end(entityId, processedRows)
  del nobatch
except Exception as e:
  print(e)
  msg = str(e).replace("'", '').replace('"', '')  
  classLog.error(batchId, entityId, msg)

In [0]:
dbutils.notebook.exit("apidumbmeterNonHome Completed Successfully")